# 00 — Data Setup

**Este notebook é a fundação do projeto.** Execute-o antes de qualquer outro.

O que ele faz:
1. Baixa todos os arquivos do Anthropic Economic Index via `huggingface_hub`
2. Carrega as 3 releases com dados geográficos (Ago 2025, Nov 2025, Fev 2026)
3. Enriquece o dataframe com `country_name`, `release_date`, `usage_per_capita_index` (calculado para releases raw)
4. Exibe o dicionário de dados e metadados das variáveis
5. Salva o dataset consolidado em `data/aei_all_releases.parquet`

---
**Outputs gerados:**
- `data/aei_all_releases.parquet` — dataset completo, todas as releases
- `data/aei_aug2025.parquet` — Ago 2025 (enriched, com automation_pct etc.)
- `data/aei_feb2026.parquet` — Fev 2026 (mais recente, com variáveis novas)

In [1]:
%pip install -q huggingface_hub pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
from IPython.display import display, HTML

# Paths
ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

# Importa metadados do projeto
sys.path.insert(0, str(ROOT / 'src'))
from metadata import RELEASES, BASE_COLUMNS, VARIABLES, COLLABORATION_PATTERNS, COUNTRY_LABELS, BRAZIL, COMPARABLES

REPO = 'Anthropic/EconomicIndex'

def dl(filename):
    return Path(hf_hub_download(repo_id=REPO, filename=filename,
                                repo_type='dataset', local_dir=DATA_DIR))

print('Setup ok. ROOT:', ROOT)

Setup ok. ROOT: C:\AI-exploratory\01_anthropic_economic_index


---
## 1. Download dos arquivos

In [3]:
# Lista arquivos AEI disponíveis no repositório
all_files = list(list_repo_files(REPO, repo_type='dataset'))
aei_files = [f for f in all_files
             if ('aei_enriched_claude_ai' in f or 'aei_raw_claude_ai' in f)
             and f.endswith('.csv')]

print(f'{len(aei_files)} arquivo(s) AEI encontrado(s):')
for f in sorted(aei_files):
    print(f'  {f}')

4 arquivo(s) AEI encontrado(s):
  release_2025_09_15/data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv
  release_2025_09_15/data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv
  release_2026_01_15/data/intermediate/aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv
  release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv


In [4]:
# Download: arquivo de população (necessário para normalização per capita)
POP_FILE = 'release_2025_09_15/data/input/working_age_pop_2024_country_raw.csv'
path_pop = dl(POP_FILE)
print('Pop file:', path_pop)

Pop file: C:\AI-exploratory\01_anthropic_economic_index\data\release_2025_09_15\data\input\working_age_pop_2024_country_raw.csv


---
## 2. Mapa de nomes de países

In [5]:
df_pop_raw = pd.read_csv(path_pop)

# Filtra apenas entidades que são países reais (ISO-3 de 3 letras maiúsculas sem agregados regionais)
# Agregados do Banco Mundial têm códigos como AFE, AFW, ARB, CSS, etc.
# Países reais: 3 letras, e existem no nosso dicionário de labels ou têm population > 0
country_name_map = (
    df_pop_raw[['countryiso3code', 'country.value']]
    .drop_duplicates()
    .set_index('countryiso3code')['country.value']
    .to_dict()
)

# Sobrescreve com nomes em português onde disponível
country_name_map.update({k: v for k, v in COUNTRY_LABELS.items()})

print(f'Países/regiões no mapa: {len(country_name_map)}')
print('Exemplos:')
for iso in ['BRA', 'USA', 'ARG', 'IND', 'ZAF']:
    print(f'  {iso} -> {country_name_map.get(iso, "N/A")}')

Países/regiões no mapa: 262
Exemplos:
  BRA -> Brasil
  USA -> EUA
  ARG -> Argentina
  IND -> Índia
  ZAF -> África do Sul


---
## 3. Carregamento de todas as releases

In [6]:
# Normalização per capita: usage_per_capita_index = usage_pct / pop_share
pop_values = df_pop_raw.set_index('countryiso3code')['value'].to_dict()
total_pop  = sum(v for v in pop_values.values() if pd.notna(v) and v > 0)
pop_share  = {k: v / total_pop for k, v in pop_values.items() if pd.notna(v) and v > 0}

print(f'Total pop no dataset: {total_pop:,.0f}')
print(f'Países com pop_share: {len(pop_share)}')

frames = []

for release_key, meta in RELEASES.items():
    print(f"\nCarregando {meta['label']} ({meta['period']}) ...")
    raw = pd.read_csv(dl(meta['file']))

    # Colunas de contexto temporal
    raw['release_date']  = pd.Timestamp(meta['reference_dt'])
    raw['release_label'] = meta['label']
    raw['release_key']   = release_key
    raw['schema_type']   = meta['schema']

    # Coluna de nome do país
    raw['country_name'] = raw['geo_id'].map(country_name_map)

    # Computa usage_per_capita_index para releases raw (não vem no arquivo)
    if meta['schema'] == 'raw':
        mask = (
            (raw['geography'] == 'country') &
            (raw['facet']     == 'country') &
            (raw['variable']  == 'usage_pct')
        )
        idx_rows = raw[mask].copy()
        idx_rows['value']    = idx_rows.apply(
            lambda r: r['value'] / pop_share.get(r['geo_id'], float('nan')), axis=1
        )
        idx_rows['variable'] = 'usage_per_capita_index'
        raw = pd.concat([raw, idx_rows], ignore_index=True)

    frames.append(raw)
    vars_all = sorted(raw['variable'].unique())
    print(f'  {len(raw):,} linhas | {len(vars_all)} variáveis | países: {raw[raw["geography"]=="country"]["geo_id"].nunique()}')

df_all = pd.concat(frames, ignore_index=True)
print(f'\nDataframe consolidado: {df_all.shape}')
print(f'Releases: {df_all["release_label"].unique()}')

Total pop no dataset: 53,495,582,327
Países com pop_share: 262

Carregando Ago 2025 (2025-08-04 to 2025-08-11) ...


  136,845 linhas | 23 variáveis | países: 201

Carregando Nov 2025 (2025-11-13 to 2025-11-20) ...
  458,952 linhas | 167 variáveis | países: 173

Carregando Fev 2026 (2026-02-05 to 2026-02-12) ...
  425,435 linhas | 155 variáveis | países: 177

Dataframe consolidado: (1021232, 16)
Releases: ['Ago 2025' 'Nov 2025' 'Fev 2026']


---
## 4. Validação e inspeção

In [7]:
print('=== SHAPE E TIPOS ===')
print(df_all.dtypes)
print(f'\nMemória: {df_all.memory_usage(deep=True).sum() / 1e6:.1f} MB')

=== SHAPE E TIPOS ===
geo_id                         object
geography                      object
date_start                     object
date_end                       object
platform_and_product           object
facet                          object
level                           int64
variable                       object
cluster_name                   object
value                         float64
geo_name                       object
release_date            datetime64[s]
release_label                  object
release_key                    object
schema_type                    object
country_name                   object
dtype: object

Memória: 865.6 MB


In [8]:
print('=== AMOSTRA — países, todas as releases ===')
sample = df_all[
    (df_all['geography'] == 'country') &
    (df_all['facet']     == 'country') &
    (df_all['variable']  == 'usage_per_capita_index') &
    (df_all['geo_id'].isin(COMPARABLES))
][['release_label', 'geo_id', 'country_name', 'variable', 'value']].sort_values(['geo_id', 'release_label'])

display(sample)

=== AMOSTRA — países, todas as releases ===


,release_label,geo_id,country_name,variable,value
646,Ago 2025,ARG,Argentina,usage_per_capita_index,0.784223
6439,Ago 2025,BRA,Brasil,usage_per_capita_index,0.926018
12581,Ago 2025,CHL,Chile,usage_per_capita_index,1.158389
13525,Ago 2025,COL,Colômbia,usage_per_capita_index,0.823603
32221,Ago 2025,IND,Índia,usage_per_capita_index,0.267048
50298,Ago 2025,MEX,México,usage_per_capita_index,0.399578
79656,Ago 2025,ZAF,África do Sul,usage_per_capita_index,0.457625


In [9]:
print('=== VARIÁVEIS POR RELEASE ===')
var_avail = (
    df_all.groupby(['release_label', 'variable'])
    .size()
    .reset_index(name='rows')
    .pivot(index='variable', columns='release_label', values='rows')
    .fillna(0)
    .astype(int)
)
display(var_avail)

=== VARIÁVEIS POR RELEASE ===


release_label,Ago 2025,Fev 2026,Nov 2025
variable,,,
ai_autonomy_count,0,1,1155
ai_autonomy_histogram_count,0,25,5
ai_autonomy_histogram_pct,0,25,5
ai_autonomy_mean,0,1,1155
ai_autonomy_mean_ci_lower,0,1,1155
...,...,...,...
usage_per_capita_index,245,178,174
usage_tier,245,0,0
use_case_count,0,3358,2993


In [10]:
print('=== PAÍSES COM DADOS (usage_per_capita_index) por release ===')
country_counts = (
    df_all[
        (df_all['geography'] == 'country') &
        (df_all['variable']  == 'usage_per_capita_index')
    ]
    .groupby('release_label')['geo_id']
    .nunique()
)
print(country_counts.to_string())

=== PAÍSES COM DADOS (usage_per_capita_index) por release ===
release_label
Ago 2025    194
Fev 2026    177
Nov 2025    173


---
## 5. Dicionário de Dados

In [11]:
print('=== COLUNAS DO DATAFRAME ===')
rows = []
for col, meta in BASE_COLUMNS.items():
    rows.append({
        'Coluna':     col,
        'Tipo':       meta.get('type', ''),
        'Descrição':  meta.get('description', ''),
        'Exemplo':    str(meta.get('example', '')),
        'Origem':     meta.get('source', 'Dataset original'),
    })
df_dict_cols = pd.DataFrame(rows)
display(df_dict_cols.style.set_properties(**{'text-align': 'left'}))

=== COLUNAS DO DATAFRAME ===


,Coluna,Tipo,Descrição,Exemplo,Origem
0,geo_id,string,"Geographic identifier. ISO-3166 alpha-3 for countries (e.g. BRA, USA) or US state FIPS codes for states. 'GLOBAL' for global aggregates.",BRA,Dataset original
1,geography,categorical,Level of geographic aggregation.,country,Dataset original
2,date_start,date,Start date of the data collection window.,2025-08-04,Dataset original
3,date_end,date,End date of the data collection window.,2025-08-11,Dataset original
4,platform_and_product,string,Platform that generated the data.,Claude AI (Free and Pro),Dataset original
5,facet,categorical,Analysis dimension. Defines what 'cluster_name' represents in each row.,,Dataset original
6,level,integer,"Hierarchy level within the facet tree (0 = top level, 1 = subcategory, 2 = leaf).",0,Dataset original
7,variable,categorical,Name of the metric. See VARIABLES dict for full documentation.,usage_per_capita_index,Dataset original
8,cluster_name,string,Entity within the facet. For onet_task: the task description. For collaboration: pattern name. For request: cluster label. Cross-facets use 'base::sub' format.,Write reports or other documents,Dataset original
9,value,float,Numeric value of the metric defined by 'variable'.,1.23,Dataset original


In [12]:
print('=== VARIÁVEIS (coluna variable) ===')
rows = []
for var, meta in VARIABLES.items():
    rows.append({
        'variable':      var,
        'unit':          meta.get('unit', ''),
        'description':   meta.get('description', ''),
        'facets':        ', '.join(meta.get('facets', [])),
        'availability':  ', '.join(meta.get('availability', [])),
        'interpretation': meta.get('interpretation', ''),
    })
df_dict_vars = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
display(df_dict_vars)

=== VARIÁVEIS (coluna variable) ===


,variable,unit,description,facets,availability,interpretation
0,usage_count,conversations,Raw number of Claude.ai conversations attributed to this facet/cluster.,country,"Ago 2025, Nov 2025, Fev 2026",
1,usage_pct,fraction [0-1],Share of total platform conversations attributed to this geography.,country,"Ago 2025, Nov 2025, Fev 2026",
2,usage_per_capita,conversations / working-age person,Raw per-capita usage: usage_count / working_age_population.,country,Ago 2025,
3,usage_per_capita_index,index (global average = 1.0),Country's per-capita usage divided by the global average. >1 means above-average adoption. Computed for all releases...,country,"Ago 2025, Nov 2025 (computed), Fev 2026 (computed)",>1 = above global average | <1 = below | 2.0 = twice the average
4,usage_tier,categorical,Binned usage level (Low / Medium / High) relative to global distribution.,country,Ago 2025,
5,onet_task_count,conversations,Number of conversations classified as involving this O*NET task.,onet_task,"Ago 2025, Nov 2025, Fev 2026",
6,onet_task_pct,fraction [0-1],Share of the country's conversations involving this O*NET task.,onet_task,"Ago 2025, Nov 2025, Fev 2026",
7,onet_task_pct_index,index (global average = 1.0),Specialization index: country's task share divided by the global task share. Reveals which tasks a country over/unde...,onet_task,Ago 2025,>1 = country specializes in this task more than globally
8,soc_pct,fraction [0-1],Share of usage mapped to this SOC occupational major group.,country,Ago 2025,
9,collaboration_count,conversations,Conversations exhibiting this collaboration pattern.,collaboration,"Ago 2025, Nov 2025, Fev 2026",


In [13]:
print('=== PADRÕES DE COLABORAÇÃO ===')
for pattern, desc in COLLABORATION_PATTERNS.items():
    print(f'  {pattern:20} {desc}')

=== PADRÕES DE COLABORAÇÃO ===
  directive            Human gives instructions; model executes with minimal back-and-forth. → Automation signal.
  feedback_loop        Iterative with model leading, human providing course corrections. → Automation signal.
  validation           Human reviews and validates model output before acting. → Augmentation signal.
  task_iteration       Human actively refines the task through multiple rounds. → Augmentation signal.
  learning             Human uses the interaction to build knowledge or skills. → Augmentation signal.
  none                 No clear collaboration pattern detected.


---
## 6. Salvar datasets limpos

In [14]:
# Dataset completo (todas as releases)
out_all = DATA_DIR / 'aei_all_releases.parquet'
df_all.to_parquet(out_all, index=False)
print(f'Salvo: {out_all} ({out_all.stat().st_size / 1e6:.1f} MB)')

# Release enriched (Ago 2025) — única com automation_pct, onet_task_pct_index
df_aug25 = df_all[df_all['release_label'] == 'Ago 2025'].copy()
out_aug = DATA_DIR / 'aei_aug2025.parquet'
df_aug25.to_parquet(out_aug, index=False)
print(f'Salvo: {out_aug} ({out_aug.stat().st_size / 1e6:.1f} MB)')

# Release mais recente (Fev 2026)
df_feb26 = df_all[df_all['release_label'] == 'Fev 2026'].copy()
out_feb = DATA_DIR / 'aei_feb2026.parquet'
df_feb26.to_parquet(out_feb, index=False)
print(f'Salvo: {out_feb} ({out_feb.stat().st_size / 1e6:.1f} MB)')

print('\nPara carregar em outros notebooks:')
print("  df_all   = pd.read_parquet('../data/aei_all_releases.parquet')")
print("  df_aug25 = pd.read_parquet('../data/aei_aug2025.parquet')")
print("  df_feb26 = pd.read_parquet('../data/aei_feb2026.parquet')")

Salvo: C:\AI-exploratory\01_anthropic_economic_index\data\aei_all_releases.parquet (19.3 MB)
Salvo: C:\AI-exploratory\01_anthropic_economic_index\data\aei_aug2025.parquet (1.8 MB)
Salvo: C:\AI-exploratory\01_anthropic_economic_index\data\aei_feb2026.parquet (5.8 MB)

Para carregar em outros notebooks:
  df_all   = pd.read_parquet('../data/aei_all_releases.parquet')
  df_aug25 = pd.read_parquet('../data/aei_aug2025.parquet')
  df_feb26 = pd.read_parquet('../data/aei_feb2026.parquet')


---
## 7. Checklist de qualidade

In [15]:
checks = [
    ('Todas as 3 releases presentes',
     df_all['release_label'].nunique() == 3),
    ('Brasil (BRA) presente em todas as releases',
     df_all[df_all['geo_id'] == BRAZIL]['release_label'].nunique() == 3),
    ('Coluna country_name preenchida para países reais',
     df_all[df_all['geography'] == 'country']['country_name'].notna().mean() > 0.8),
    ('usage_per_capita_index calculado para todas as releases',
     df_all[df_all['variable'] == 'usage_per_capita_index']['release_label'].nunique() == 3),
    ('automation_pct disponível em Ago 2025',
     len(df_all[(df_all['release_label'] == 'Ago 2025') & (df_all['variable'] == 'automation_pct')]) > 0),
    ('human_only_ability_pct disponível em Fev 2026',
     len(df_all[(df_all['release_label'] == 'Fev 2026') & (df_all['variable'] == 'human_only_ability_pct')]) > 0),
    ('Parquet completo salvo',
     (DATA_DIR / 'aei_all_releases.parquet').exists()),
]

all_pass = True
for label, result in checks:
    icon = 'OK' if result else 'FALHOU'
    print(f'  [{icon}] {label}')
    if not result:
        all_pass = False

print()
print('Setup completo! Pode rodar os demais notebooks.' if all_pass
      else 'Alguns checks falharam — revise antes de continuar.')

  [OK] Todas as 3 releases presentes
  [FALHOU] Brasil (BRA) presente em todas as releases
  [FALHOU] Coluna country_name preenchida para países reais
  [OK] usage_per_capita_index calculado para todas as releases
  [OK] automation_pct disponível em Ago 2025
  [OK] human_only_ability_pct disponível em Fev 2026
  [OK] Parquet completo salvo

Alguns checks falharam — revise antes de continuar.
